# Est-ce qu'on peut deviner l'Elo avec les coups ?

Je vais essayer de prédire l'Elo d'un joueur à partir de ses coups et de sa pendule. Une ligne correspond à un joueur dans une partie.

L'idée est de faire les étapes dans l'ordre et de voir ce qui marche (ou pas). Je commence avec des choses assez simples.

Il faut avoir fait `uv sync` et choisir le kernel du projet. Le gros fichier PGN doit être à la racine. Je ne le télécharge pas ici.

## Avant de commencer

Pour chaque ligne, j'utilise les coups et les temps d'un joueur. Ce que je veux retrouver, c'est son Elo Lichess.

Je ne donne pas au modèle l'Elo de l'adversaire ni le résultat de la partie : ça donnerait trop d'indices. Je garde aussi les joueurs de test à part. Sinon, le même joueur pourrait se retrouver dans les deux groupes.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RACINE = Path.cwd().resolve()
while not (RACINE / "pyproject.toml").exists() and RACINE != RACINE.parent:
    RACINE = RACINE.parent
SRC = RACINE / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from elo import features, nettoyage
from utils.pgn import open_pgn, read_games

In [ ]:
# Les fichiers intermédiaires sont rangés à part du PGN d'origine.
FICHIER_PGN = RACINE / "lichess_db_standard_rated_2026-09.pgn.zst"
DOSSIER_DATA = RACINE / "Data"
DOSSIER_DATA.mkdir(exist_ok=True)

if not FICHIER_PGN.exists():
    raise FileNotFoundError(f"PGN introuvable : {FICHIER_PGN}. Vérifiez le chemin avant de continuer.")

print("Dossier du projet :", RACINE)
print("Fichier trouvé :", FICHIER_PGN.exists())

## Regarder le fichier

Un PGN contient des infos sur la partie puis les coups. Le fichier est compressé, donc je regarde juste quelques lignes.

Lire beaucoup de parties prend un peu de temps et crée des fichiers dans `Data/`. Pour tester, on peut mettre `50_000` au lieu de 2 millions.

In [ ]:
with open_pgn(FICHIER_PGN) as fichier:
    apercu = [next(fichier).rstrip() for _ in range(5)]

print("\n".join(apercu))
N_PARTIES_A_LIRE = 2_000_000  # réduire à 50_000 pour un premier essai

## Charger les parties

Je garde les parties en 3 minutes, avec ou sans 2 secondes d'incrément. Les deux cadences ne sont pas exactement pareilles, donc je garde cette info pour la suite.

In [ ]:
cadences_voulues = {"180+0", "180+2"}
parties_brutes = read_games(
    FICHIER_PGN,
    n_read=N_PARTIES_A_LIRE,
    time_controls=cadences_voulues,
)

print(parties_brutes.height, "parties gardées")
display(parties_brutes.select("White", "Black", "WhiteElo", "BlackElo", "TimeControl", "n_moves").head())
parties_brutes.write_parquet(DOSSIER_DATA / "pipeline_debutant_parties.parquet")

On regarde combien il y en a dans chaque cadence.

In [ ]:
comptes_cadence = parties_brutes.group_by("TimeControl").len().sort("TimeControl")
plt.bar(comptes_cadence["TimeControl"].to_list(), comptes_cadence["len"].to_list())
plt.title("Parties par cadence")
plt.ylabel("Nombre")
plt.show()

## Un peu de nettoyage

Une partie donne deux lignes : une pour les Blancs, une pour les Noirs. J'enlève les bots, les parties très courtes et quelques cas où l'Elo ou la pendule semblent peu fiables. Les règles sont dans les fonctions du projet.

Ça ne rend pas les données parfaites, mais au moins on sait à peu près sur quelles parties on travaille.

In [ ]:
parties_propres = nettoyage.filtre_parties(parties_brutes)
joueurs = nettoyage.une_ligne_par_joueur(parties_propres)
joueurs = nettoyage.filtre_joueurs(joueurs)

print("Avant nettoyage :", parties_brutes.height, "parties")
print("Après nettoyage :", parties_propres.height, "parties")
print("Lignes joueur-partie :", joueurs.height)
display(joueurs.select("joueur", "couleur", "elo", "cadence", "san", "clk").head(3))
joueurs.write_parquet(DOSSIER_DATA / "pipeline_debutant_joueurs.parquet")

## Transformer les coups en nombres

Le modèle ne sait pas quoi faire directement d'une liste comme `e4, Nf3, Bb5`. Je résume donc la partie avec quelques nombres : nombre de coups, temps utilisé, captures, etc.

Ça ne mesure pas vraiment si les coups sont bons. Ce sont juste des indices faciles à calculer.

In [ ]:
table_features = features.calcule_features(joueurs)
table_features = table_features.with_columns(
    pl.col("cadence").str.extract(r"\+(\d+)").cast(pl.Float64).alias("increment_s")
)

print("Lignes :", table_features.height)
print("Colonnes :", table_features.width)
display(table_features.select("elo", "n_coups", "t_ouverture", "pct_rapides", "cv_temps", "pct_captures", "temps_restant_fin", "increment_s").head())
table_features.write_parquet(DOSSIER_DATA / "pipeline_debutant_features.parquet")

## Quelques graphiques

Je regarde d'abord à quoi ressemblent les Elo et quelques variables. Une tendance sur un graphique ne veut pas forcément dire que l'une cause l'autre.

In [ ]:
plt.hist(table_features["elo"].to_numpy(), bins=30)
plt.title("Elo")
plt.xlabel("Elo")
plt.ylabel("Nombre de lignes")
plt.show()
print("Elo du milieu :", table_features["elo"].median())

In [ ]:
# J'en prends un bout pour que le graphique reste lisible.
echantillon = table_features.sample(n=min(6_000, table_features.height), seed=42)
fig, axes = plt.subplots(1, 2)

axes[0].scatter(echantillon["elo"].to_numpy(), echantillon["t_ouverture"].to_numpy(), alpha=0.2, s=12)
axes[0].set_title("Temps au début")

axes[1].scatter(echantillon["elo"].to_numpy(), echantillon["pct_rapides"].to_numpy(), alpha=0.2, s=12)
axes[1].set_title("Coups rapides")

plt.show()

In [ ]:
FEATURES = ["n_coups", "t_ouverture", "pct_rapides", "cv_temps", "pct_captures", "temps_restant_fin", "increment_s"]
taux_manquants = sorted(
    [(nom, table_features[nom].null_count() / table_features.height * 100) for nom in FEATURES],
    key=lambda element: element[1],
)

plt.barh([nom for nom, _ in taux_manquants], [taux for _, taux in taux_manquants])
plt.title("Valeurs manquantes (%)")
plt.show()

## Séparer train et test

Je mets 20 % des joueurs de côté pour le test. Toutes les parties d'un joueur restent ensemble. Comme ça, le modèle est testé sur des joueurs qu'il n'a pas vus à l'entraînement.

Je vais aussi faire une prédiction très simple : donner à tout le monde l'Elo médian du train. Ça servira de point de comparaison.

In [ ]:
# Le nom du joueur sert au découpage, pas à la prédiction.
donnees = table_features.filter(pl.col("elo").is_not_null())
decoupe = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
indices_train, indices_test = next(decoupe.split(donnees, groups=donnees["joueur"].to_list()))
train = donnees[indices_train]
test = donnees[indices_test]

joueurs_train = set(train["joueur"].to_list())
joueurs_test = set(test["joueur"].to_list())
assert joueurs_train.isdisjoint(joueurs_test)
print("Train :", train.height, "lignes et", len(joueurs_train), "joueurs")
print("Test :", test.height, "lignes et", len(joueurs_test), "joueurs")

## Entraîner un premier modèle

La régression linéaire combine les variables avec des poids. Elle est plutôt facile à comprendre, mais elle ne sait pas très bien représenter des relations compliquées.

Il y a des cases vides, par exemple si une partie n'est pas allée jusqu'à la finale. Je les remplace par la médiane du train. Je standardise aussi les nombres pour qu'ils soient à peu près à la même échelle.

In [ ]:
X_train = train.select(FEATURES).to_numpy()
y_train = train["elo"].to_numpy()
X_test = test.select(FEATURES).to_numpy()
y_test = test["elo"].to_numpy()

modele = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    LinearRegression(),
)
modele.fit(X_train, y_train)
predictions = modele.predict(X_test)

prediction_mediane = np.full(len(y_test), np.median(y_train))
mae_mediane = mean_absolute_error(y_test, prediction_mediane)
mae_lineaire = mean_absolute_error(y_test, predictions)
r2_lineaire = r2_score(y_test, predictions)

print("Erreur avec la médiane :", round(mae_mediane), "points")
print("Erreur du modèle :", round(mae_lineaire), "points")
print("R2 :", round(r2_lineaire, 2))

La MAE donne l'erreur moyenne en points Elo. Plus elle est basse, mieux c'est. Le R2 est un autre score ; proche de 1 c'est bien, autour de 0 le modèle n'apporte pas grand-chose.

In [ ]:
fig, axes = plt.subplots(1, 2)

axes[0].scatter(y_test, predictions, alpha=0.2, s=12)
axes[0].plot([500, 3000], [500, 3000], "r--")
axes[0].set_title("Réel et prédit")

erreurs = predictions - y_test
axes[1].hist(erreurs, bins=40, color="darkorange", edgecolor="white")
axes[1].axvline(0, color="black", linestyle="--")
axes[1].set_title("Erreurs")

plt.show()

In [ ]:
resultats = pl.DataFrame({"elo_reel": y_test, "elo_predit": predictions})
erreurs_par_tranche = (
    resultats
    .with_columns(
        (pl.col("elo_reel") // 200 * 200).alias("tranche_elo"),
        (pl.col("elo_reel") - pl.col("elo_predit")).abs().alias("erreur_absolue"),
    )
    .group_by("tranche_elo")
    .agg(pl.col("erreur_absolue").mean().alias("MAE"), pl.len().alias("nombre"))
    .sort("tranche_elo")
)

plt.bar(erreurs_par_tranche["tranche_elo"].to_list(), erreurs_par_tranche["MAE"].to_list(), width=160)
plt.title("Erreur selon l'Elo")
plt.ylabel("MAE")
plt.show()
display(erreurs_par_tranche)

## Un exemple

Je prends une ligne du test pour voir ce que ça donne. L'Elo réel est affiché juste pour comparer.

In [ ]:
exemple = test.select("joueur", "elo", *FEATURES).head(1)
elo_predit = modele.predict(exemple.select(FEATURES).to_numpy())[0]
print("Elo réel :", exemple["elo"][0])
print("Elo prédit :", round(elo_predit))

## Pour finir

Une partie ne suffit pas toujours pour estimer le niveau : il y a du hasard et les coups dépendent aussi de l'adversaire. La régression est simple, donc elle ne va pas tout capturer.

Une idée pour la suite serait de faire la moyenne sur plusieurs parties, ou d'essayer d'estimer si les coups sont bons. Il faudra comparer avec la baseline et garder le même découpage des joueurs.